In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

df = pd.read_csv("D:\Python Courses\Machine-Learning-Projects\House Price Prediction\House Prediction.csv")
print(df.head())
print(df.info())
print(df.isnull().sum())

                  date      price  bedrooms  ...       city  statezip  country
0  2014-05-02 00:00:00   313000.0       3.0  ...  Shoreline  WA 98133      USA
1  2014-05-02 00:00:00  2384000.0       5.0  ...    Seattle  WA 98119      USA
2  2014-05-02 00:00:00   342000.0       3.0  ...       Kent  WA 98042      USA
3  2014-05-02 00:00:00   420000.0       3.0  ...   Bellevue  WA 98008      USA
4  2014-05-02 00:00:00   550000.0       4.0  ...    Redmond  WA 98052      USA

[5 rows x 18 columns]
<class 'pandas.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   str    
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterf

In [2]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
print(df.columns)

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country', 'year', 'month'],
      dtype='str')


In [3]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')
ohe_columns= ohe.fit_transform(df[['city','statezip']])
print(ohe_columns.columns)

Index(['city_Algona', 'city_Auburn', 'city_Beaux Arts Village',
       'city_Bellevue', 'city_Black Diamond', 'city_Bothell', 'city_Burien',
       'city_Carnation', 'city_Clyde Hill', 'city_Covington',
       ...
       'statezip_WA 98155', 'statezip_WA 98166', 'statezip_WA 98168',
       'statezip_WA 98177', 'statezip_WA 98178', 'statezip_WA 98188',
       'statezip_WA 98198', 'statezip_WA 98199', 'statezip_WA 98288',
       'statezip_WA 98354'],
      dtype='str', length=121)


In [4]:
new_df = df.drop(columns=['date','street','city','statezip','country','price','country'])
print(new_df.columns)

Index(['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
       'yr_built', 'yr_renovated', 'year', 'month'],
      dtype='str')


In [5]:
feature_columns = np.concatenate((new_df,ohe_columns),axis=1)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x = scaler.fit_transform(feature_columns)
y = df['price']

In [7]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [16]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
learning_rate=  0.1, max_depth = 2, min_samples_leaf =  2, min_samples_split = 2, n_estimators = 300
)

model.fit(x_train, y_train)

y_pred = model.predict(x_test)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"MAE : {mean_absolute_error(y_test,y_pred)}")
print(f"MSE : {mean_squared_error(y_test,y_pred)}")
print(f"R2 : {r2_score(y_test,y_pred)}")

MAE : 148998.91053999998
MSE : 960723421446.999
R2 : 0.05797237908890729


In [9]:
print(df.columns)

Index(['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city',
       'statezip', 'country', 'year', 'month'],
      dtype='str')


In [10]:
bedrooms = float(input("Enter Bedrooms: "))
bathrooms = float(input("Enter Bathrooms: "))
sqft_living = float(input("Enter sqft_living: "))
sqft_lot = float(input('Enter sqft_lot'))
floors = float(input("Enter Floors: "))
waterfront = int(input("Waterfront(1/0): "))
view = int(input("View(1/0): "))
condition = float(input("Enter Condition: "))
sqft_above = float(input("Enter sqft_above: "))
sqft_basement = float(input("Enter sqft_basement: "))
yr_built = float(input("Enter yr_built: "))
yr_renovated = float(input("Enter yr_renovated: "))
year = float(input("Enter Year Of Sellig: "))
month = float(input("Enter Month Of Sellig: "))
city = input("Enter City: ")
statezip = input("Enter Statezip: ")


user_input_columns = np.array([[bedrooms,bathrooms,sqft_living,sqft_lot,floors,
                              waterfront,view,condition,sqft_above,sqft_basement,
                              yr_built,yr_renovated,year,month]])
string_inputs = ohe.transform([[city,statezip]]).to_numpy()
concate_df = np.concatenate((user_input_columns,string_inputs),axis=1)
scaled_concate_df = scaler.transform(concate_df)

predict_price = model.predict(scaled_concate_df)

print(f"Predicted Price is: {predict_price[0]:,.2f}")



ValueError: could not convert string to float: ''

In [15]:
"""Hyper Parameter Tunning"""

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

n_estimators = [100, 200, 300]

learning_rate = [0.05, 0.1, 0.2]

max_depth = [2, 3, 4]

min_samples_split = [2, 5]

min_samples_leaf = [1, 2]

param_grid = {

    'n_estimators': n_estimators,
    'learning_rate': learning_rate,
    'max_depth': max_depth,
    'min_samples_split': min_samples_split,
    'min_samples_leaf': min_samples_leaf

}
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best R2:", grid.best_score_)

Best Parameters: {'learning_rate': 0.1, 'max_depth': 2, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}
Best R2: 0.648185312297524


In [17]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

best_params = grid.best_params_

model = GradientBoostingRegressor(
    **best_params,
    random_state=42
)

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print("Manual Test R2:", r2_score(y_test, y_pred))
print("Grid CV R2:", grid.best_score_)

Manual Test R2: 0.05799913169249693
Grid CV R2: 0.648185312297524


AttributeError: 'numpy.ndarray' object has no attribute 'columns'